# Multilingual and Cross-lingual Prompting

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/prompt-engineering/multilingual-prompting.ipynb)

## Overview

This tutorial explores the concepts and techniques of multilingual and cross-lingual prompting using open-source large language models running locally. We use **Qwen3-8B**, which natively supports **29+ languages** including Chinese, English, French, Spanish, Portuguese, German, Italian, Russian, Japanese, Korean, Vietnamese, Thai, and Arabic.

## Motivation

As AI language models become increasingly sophisticated, there is a growing need to leverage their capabilities across linguistic boundaries. With open-source multilingual models like Qwen3 now rivaling proprietary APIs, we can build inclusive and globally accessible AI applications entirely locally — with no API keys, no usage fees, and full control over the model.

## Key Components

1. **Multilingual Prompt Design** — Strategies for creating prompts that work effectively in multiple languages.
2. **Language Detection and Adaptation** — Techniques for identifying the input language and adapting the model's response accordingly.
3. **Cross-lingual Translation** — Methods for using language models to perform translation tasks between different languages.
4. **Chat-Templated Prompting** — Using the model's native chat template for creating flexible, language-aware prompts.
5. **Handling Non-Latin Scripts** — Considerations and techniques for working with languages that use non-Latin alphabets.

## Method Details

We use the **Qwen3-8B** model via HuggingFace Transformers, loaded in 4-bit quantization so it fits within a free Google Colab GPU. Our approach includes:

1. Setting up the environment with HuggingFace Transformers and loading the model.
2. Creating multilingual prompts using Python f-strings and the chat template.
3. Implementing language detection and response adaptation.
4. Designing prompts for cross-lingual translation tasks.
5. Handling various writing systems and scripts.
6. Exploring techniques for improving translation quality and cultural sensitivity.

Throughout the tutorial, we provide examples in multiple languages to illustrate the concepts and techniques discussed.

## Conclusion

By the end of this tutorial, you will have gained practical skills in designing and implementing multilingual and cross-lingual prompts with an open-source model. These techniques enable you to create inclusive and globally accessible AI applications — running entirely on your own hardware — leveraging Qwen3's native multilingual capabilities across diverse linguistic contexts.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **Overview**
- Understand **Motivation**
- Understand **Key Components**
- Understand **Method Details**
- Understand **Conclusion**


## How Multilingual Models Work: Shared Embedding Space

### The Core Insight: One Vector Space for All Languages

Multilingual models like Qwen3 don't have separate "English mode" and "Chinese mode." Instead, they map tokens from **every** language into a **single shared embedding space** — a high-dimensional vector space (typically 1024–5120 dimensions depending on model size).

During pretraining on massive multilingual corpora, the model observes:
- The same concepts described in different languages
- Parallel and comparable texts across languages
- Shared vocabulary items (numbers, proper nouns, borrowed words)

Through gradient descent, semantically equivalent words across languages are pushed **closer together** in this space. For example:
- `"cat"` (English), `"猫"` (Chinese), `"gato"` (Spanish) → nearby vectors
- `"cat"` (English) and `"democracy"` (English) → distant vectors

This is fundamentally different from having separate models per language. The shared space enables **cross-lingual transfer**: knowledge learned from English text helps the model understand Chinese, and vice versa.

### Why Does This Work?

The key mechanisms are:

1. **Shared subword tokens** — Many languages share numeric digits, punctuation, Latin-script loanwords, and technical terms. These act as "anchor points" that tie the embedding spaces together.

2. **Contextual similarity** — Words that appear in similar contexts across languages (e.g., "king" appears near "queen" and "throne" in English, just as "国王" appears near "女王" and "王座" in Chinese) develop similar embeddings.

3. **Attention patterns** — The transformer attention mechanism learns language-agnostic relationships (subject-verb-object patterns, modifier-noun relationships) that generalize across languages.

In [ ]:
# --- Shared Embedding Space: Cosine Similarity Across Languages ---
import numpy as np

def get_token_embedding(word):
    """Look up the embedding for the first token of a word."""
    token_ids = tokenizer.encode(word, add_special_tokens=False)
    # Use the first subword token as the representative
    first_id = token_ids[0]
    # Access the raw embedding weights from the model
    embedding = model.model.embed_tokens.weight[first_id].detach().cpu().float().numpy()
    return embedding, token_ids

def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Words for "cat" in multiple languages
cat_words = {
    "English":  "cat",
    "Chinese":  "猫",
    "Spanish":  "gato",
    "French":   "chat",
    "Japanese": "猫",    # same character as Chinese
    "Korean":   "고양이",
    "German":   "Katze",
}

# An unrelated word for contrast
unrelated_words = {
    "democracy": "democracy",
    "mountain":  "mountain",
    "algorithm": "algorithm",
}

# Collect embeddings
cat_embeddings = {}
for lang, word in cat_words.items():
    emb, ids = get_token_embedding(word)
    cat_embeddings[lang] = emb
    tokens_str = tokenizer.convert_ids_to_tokens(ids)
    print(f"  {lang:10s} '{word}' → token(s): {tokens_str}")

unrelated_embeddings = {}
for label, word in unrelated_words.items():
    emb, _ = get_token_embedding(word)
    unrelated_embeddings[label] = emb

print("\n--- Cosine Similarity: 'cat' across languages ---")
langs = list(cat_embeddings.keys())
for i in range(len(langs)):
    for j in range(i + 1, len(langs)):
        sim = cosine_similarity(cat_embeddings[langs[i]], cat_embeddings[langs[j]])
        print(f"  {langs[i]:10s} ↔ {langs[j]:10s} : {sim:.4f}")

print("\n--- Cosine Similarity: 'cat' (English) vs unrelated words ---")
for label, emb in unrelated_embeddings.items():
    sim = cosine_similarity(cat_embeddings["English"], emb)
    print(f"  cat ↔ {label:12s} : {sim:.4f}")

print("\n→ Semantically equivalent words across languages should show HIGHER")
print("  similarity than unrelated words in the SAME language.")

### What the Similarity Scores Tell Us

The cosine similarity results above demonstrate a fundamental property of multilingual models:

- **Cross-lingual pairs** (e.g., English "cat" ↔ Chinese "猫") show **moderate to high** similarity because pretraining pushes semantically equivalent concepts toward the same region of embedding space.
- **Unrelated pairs** (e.g., "cat" ↔ "democracy") show **lower** similarity, even within the same language.

> **Important caveat:** These are *static* (input) embeddings — the first layer of the model. The real magic happens in the **contextualized** representations produced by the transformer layers. By the final layer, the representations for equivalent concepts across languages converge even more strongly. The input embeddings are just the starting point.

This shared representation is why translation "works" without ever being explicitly trained as a seq2seq translation model. The model maps between languages because they already share a representation space.

## Setup

Install dependencies, mount Google Drive for model caching, and load **Qwen3-8B** in 4-bit quantization.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Qwen3-8B supports 29+ languages including Chinese, English, French,
# Spanish, Portuguese, German, Italian, Russian, Japanese, Korean, Vietnamese, Thai, Arabic
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None, do_sample=do_sample, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

## Tokenization Across Scripts

### Why Tokenization Matters for Multilingual Prompting

Before any text reaches the model's transformer layers, it must be **tokenized** — split into subword units from the model's fixed vocabulary. This seemingly mechanical step has profound implications for multilingual applications:

- **Cost**: Many APIs charge per token. More tokens = higher cost.
- **Context window**: The model has a fixed context length (in tokens, not characters). Languages that tokenize less efficiently consume more of this window.
- **Processing speed**: More tokens = more computation = slower inference.

Different writing systems tokenize very differently. Let's see this empirically.

In [ ]:
# --- Tokenization Across Scripts: Comparative Analysis ---

sentences = {
    "English":  "Artificial intelligence is transforming the world",
    "Chinese":  "人工智能正在改变世界",
    "Arabic":   "الذكاء الاصطناعي يحول العالم",
    "Hindi":    "कृत्रिम बुद्धिमत्ता दुनिया को बदल रही है",
    "Japanese": "人工知能は世界を変えている",
    "Korean":   "인공지능이 세계를 변화시키고 있다",
}

print("=" * 80)
print(f"{'Language':<11} {'Chars':>5} {'Tokens':>6} {'Chars/Tok':>9}  Token breakdown")
print("=" * 80)

for lang, sent in sentences.items():
    token_ids = tokenizer.encode(sent, add_special_tokens=False)
    tokens = tokenizer.convert_ids_to_tokens(token_ids)
    n_chars = len(sent)
    n_tokens = len(token_ids)
    chars_per_tok = n_chars / n_tokens if n_tokens > 0 else 0

    # Truncate token display if too long
    token_str = str(tokens)
    if len(token_str) > 60:
        token_str = token_str[:57] + "..."

    print(f"{lang:<11} {n_chars:>5} {n_tokens:>6} {chars_per_tok:>9.2f}  {token_str}")

print()
print("--- Detailed Token Breakdown ---")
for lang, sent in sentences.items():
    token_ids = tokenizer.encode(sent, add_special_tokens=False)
    tokens = tokenizer.convert_ids_to_tokens(token_ids)
    print(f"\n{lang}: \"{sent}\"")
    for i, (tid, tok) in enumerate(zip(token_ids, tokens)):
        decoded = tokenizer.decode([tid])
        print(f"  [{i:2d}] id={tid:>6d}  token='{tok}'  → decoded='{decoded}'")

### What Token Efficiency Tells Us

The results above reveal a critical asymmetry in multilingual models:

| Observation | Explanation |
|---|---|
| **English is most token-efficient** | The tokenizer's vocabulary is built from training data, which is overwhelmingly English. English words map to single or few tokens. |
| **CJK scripts (Chinese, Japanese, Korean)** | Each character often becomes 1-2 tokens. Despite fewer characters, CJK text may use comparable token counts to English. |
| **Arabic** | Right-to-left script with complex morphology. Words are often split into many subword pieces. |
| **Hindi (Devanagari)** | Each syllable may require multiple tokens since Devanagari has fewer vocabulary entries. |

### Practical Implications

1. **Context window budgeting**: A 4096-token context holds roughly ~3000 English words but significantly fewer Hindi or Arabic words. Plan your prompts accordingly.
2. **Prompt compression**: For non-English languages, keep prompts concise. Move instructions to English where possible (see "Cross-Lingual Transfer" below).
3. **Cost estimation**: If using a paid API, non-English queries can cost 1.5–3x more than equivalent English queries.

## Multilingual Prompt Design

Let's start by creating a multilingual greeting prompt that adapts to different languages. Qwen3 natively supports all of these languages, so it can generate fluent responses without any special configuration.

In [ ]:
languages = ["English", "Spanish", "French", "German", "Japanese"]

for lang in languages:
    messages = [
        {"role": "system", "content": "You are a helpful multilingual assistant."},
        {"role": "user", "content": f"Greet the user in {lang} and provide a short introduction about the weather in a country where this language is spoken."}
    ]
    response = generate(messages)
    print(f"{lang}:")
    print(response)
    print()

## Language Detection and Adaptation

Now, let's create a prompt that detects the input language and responds accordingly. We also define reusable helper functions for **language detection** and **translation** that leverage Qwen3's native multilingual capability.

In [ ]:
# --- Reusable helpers: translation & language detection ---

def translate(text, source_lang, target_lang):
    """Translate text between any of Qwen3's 29+ supported languages."""
    messages = [
        {"role": "system", "content": f"You are a professional translator from {source_lang} to {target_lang}. Preserve meaning and cultural nuances."},
        {"role": "user", "content": text}
    ]
    return generate(messages, temperature=0.3)

def detect_language(text):
    """Detect the language of a given text."""
    messages = [
        {"role": "system", "content": "Identify the language of the following text. Respond with only the language name."},
        {"role": "user", "content": text}
    ]
    return generate(messages, max_new_tokens=20, temperature=0.1)

# --- Language-adaptive response ---

inputs = [
    "Hello, how are you?",
    "Hola, ¿cómo estás?",
    "Bonjour, comment allez-vous ?",
    "こんにちは、お元気ですか？",
    "Здравствуйте, как дела?"
]

for user_input in inputs:
    detected = detect_language(user_input)
    messages = [
        {"role": "system", "content": f"Detect the language of the user's input and respond in the same language ({detected})."},
        {"role": "user", "content": user_input}
    ]
    response = generate(messages)
    print(f"Input: {user_input}")
    print(f"Detected: {detected}")
    print(f"Response: {response}")
    print()

## Cross-lingual Translation

Let's use the `translate()` helper to perform cross-lingual translation across several language pairs.

In [ ]:
translations = [
    {"source_lang": "English", "target_lang": "French", "text": "The quick brown fox jumps over the lazy dog."},
    {"source_lang": "Spanish", "target_lang": "German", "text": "La vida es bella."},
    {"source_lang": "Japanese", "target_lang": "English", "text": "桜の花が満開です。"}
]

for t in translations:
    result = translate(t["text"], t["source_lang"], t["target_lang"])
    print(f"From {t['source_lang']} to {t['target_lang']}:")
    print(f"Original: {t['text']}")
    print(f"Translation: {result}")
    print()

## Cross-Lingual Transfer: Leveraging English Dominance

### The Resource Imbalance Problem

Not all languages are created equal in the eyes of an LLM. Model performance is strongly correlated with the amount of training data available:

| Tier | Languages | Approx. % of Training Data |
|---|---|---|
| **High-resource** | English, Chinese | 60–70% |
| **Medium-resource** | French, German, Spanish, Japanese, Korean | 15–25% |
| **Lower-resource** | Hindi, Thai, Vietnamese, Arabic | 5–10% |
| **Low-resource** | Many African, Indigenous languages | < 1% |

This means the model's "understanding" is deepest in English. But we can **exploit** this through cross-lingual transfer — using English as an intermediary to boost performance in other languages.

In [ ]:
# --- Cross-Lingual Transfer: Same Question, Different Languages ---
# Ask the same factual question in English vs. a lower-resource language
# and compare the depth and accuracy of the responses.

question_pairs = [
    {
        "topic": "Quantum Computing",
        "english": "Explain quantum entanglement in simple terms. What makes it different from classical correlations?",
        "hindi": "क्वांटम एंटैंगलमेंट को सरल शब्दों में समझाइए। यह शास्त्रीय सहसंबंधों से कैसे भिन्न है?",
        "lang": "Hindi"
    },
    {
        "topic": "Economics",
        "english": "What causes inflation and how do central banks typically respond?",
        "thai": "อะไรเป็นสาเหตุของเงินเฟ้อ และธนาคารกลางมักตอบสนองอย่างไร?",
        "lang": "Thai"
    },
]

for pair in question_pairs:
    print(f"{'='*70}")
    print(f"Topic: {pair['topic']}")
    print(f"{'='*70}")

    # English version
    msgs_en = [
        {"role": "system", "content": "You are a knowledgeable assistant. Give a clear, detailed answer."},
        {"role": "user", "content": pair["english"]}
    ]
    resp_en = generate(msgs_en, max_new_tokens=300, temperature=0.3)
    print(f"\n[ENGLISH question]\n{pair['english']}\n\n[ENGLISH response]\n{resp_en}")

    # Non-English version
    other_lang = pair["lang"]
    other_key = other_lang.lower()
    other_q = pair.get(other_key, "")
    msgs_other = [
        {"role": "system", "content": f"You are a knowledgeable assistant. Answer in {other_lang}."},
        {"role": "user", "content": other_q}
    ]
    resp_other = generate(msgs_other, max_new_tokens=300, temperature=0.3)
    print(f"\n[{other_lang.upper()} question]\n{other_q}\n\n[{other_lang.upper()} response]\n{resp_other}")
    print()

print("→ Compare: English answers are often more detailed and precise,")
print("  especially for technical topics, because the model has seen more")
print("  English training data on these subjects.")

In [ ]:
# --- English-First Prompting: Best of Both Worlds ---
# Strategy: Give instructions in English (for better comprehension)
# but request output in the target language (for the end user).

target_languages = {
    "Vietnamese": "Vietnamese",
    "Arabic": "Arabic",
    "Korean": "Korean",
}

task = "Explain three key differences between machine learning and traditional programming. Use numbered points."

for lang_name, lang in target_languages.items():
    print(f"{'='*60}")
    print(f"Target language: {lang_name}")
    print(f"{'='*60}")

    # Approach 1: Everything in the target language
    native_prompt = {
        "Vietnamese": "Giải thích ba điểm khác biệt chính giữa học máy và lập trình truyền thống. Sử dụng các điểm được đánh số.",
        "Arabic": "اشرح ثلاثة اختلافات رئيسية بين التعلم الآلي والبرمجة التقليدية. استخدم نقاطاً مرقمة.",
        "Korean": "머신러닝과 전통적인 프로그래밍의 세 가지 주요 차이점을 설명하세요. 번호가 매겨진 항목을 사용하세요.",
    }

    msgs_native = [
        {"role": "system", "content": f"You are a helpful assistant. Respond in {lang_name}."},
        {"role": "user", "content": native_prompt[lang_name]}
    ]
    resp_native = generate(msgs_native, max_new_tokens=400, temperature=0.3)
    print(f"\n[Native prompt → {lang_name} response]")
    print(resp_native)

    # Approach 2: English instruction, target language output
    msgs_english_first = [
        {"role": "system", "content": f"You are a helpful assistant. The user will give instructions in English. You MUST respond entirely in {lang_name}."},
        {"role": "user", "content": task}
    ]
    resp_eng_first = generate(msgs_english_first, max_new_tokens=400, temperature=0.3)
    print(f"\n[English prompt → {lang_name} response (English-first strategy)]")
    print(resp_eng_first)
    print()

print("→ The English-first approach often produces more structured, detailed output")
print("  because the model comprehends English instructions more precisely.")
print("  The output language is controlled during generation, which is a separate")
print("  mechanism from instruction comprehension.")

### Why English-First Prompting Works

The effectiveness of this technique comes from the model's architecture:

1. **Comprehension phase** (processing the prompt): The transformer processes input tokens through its layers. English tokens activate the **strongest, most refined** attention patterns because the model has seen orders of magnitude more English during training.

2. **Generation phase** (producing output): Once the model has "understood" the task, it generates tokens autoregressively. The output language is determined by:
   - The system prompt instruction ("Respond in Vietnamese")
   - The beginning of the response (if it starts generating Vietnamese tokens, it continues in Vietnamese)

These two phases are largely independent. You can have **precise comprehension** (via English instructions) and **fluent output** (in the target language) simultaneously.

> **When to use English-first prompting:**
> - Complex reasoning tasks (math, logic, analysis)
> - Technical explanations requiring precise terminology
> - Structured output (lists, tables, JSON)
>
> **When native-language prompting is better:**
> - Cultural or region-specific content
> - Creative writing with native idioms
> - When the end user writes the prompt directly

## Handling Non-Latin Scripts

Let's create a prompt that works with non-Latin scripts and provides transliteration. Qwen3 handles CJK, Cyrillic, Devanagari, and Arabic scripts natively.

In [ ]:
non_latin_texts = [
    {"text": "こんにちは、世界", "script": "Japanese"},
    {"text": "Здравствуй, мир", "script": "Cyrillic"},
    {"text": "नमस्ते दुनिया", "script": "Devanagari"}
]

for item in non_latin_texts:
    messages = [
        {"role": "system", "content": "You are a linguistics expert. Provide concise, structured information about the given text."},
        {"role": "user", "content": (
            f"Provide the following information for this text:\n"
            f"1. The original text\n"
            f"2. The name of the script/writing system\n"
            f"3. A transliteration to Latin alphabet\n"
            f"4. An English translation\n\n"
            f"Text: {item['text']}\nScript: {item['script']}"
        )}
    ]
    response = generate(messages)
    print(response)
    print()

## Code-Switching and Mixed Languages

### What is Code-Switching?

**Code-switching** is the practice of alternating between two or more languages within a single conversation or even a single sentence. It's a natural phenomenon in multilingual communities:

- **Spanglish**: "Vamos al store a comprar groceries" (Spanish + English)
- **Hinglish**: "Yaar, mujhe ek meeting schedule karni hai" (Hindi + English)
- **Chinese-English**: "这个project的deadline是下周" (Chinese + English)

Code-switching is common in technical contexts where domain-specific terms (API, machine learning, blockchain) are borrowed from English regardless of the base language. How well do multilingual models handle this?

In [ ]:
# --- Code-Switching: Handling Mixed-Language Input ---

mixed_inputs = [
    {
        "label": "Chinese + English (tech context)",
        "text": "这个machine learning model的performance怎么optimize？需要考虑overfitting的问题。",
        "task": "Understand this mixed Chinese-English text about ML and provide a helpful response in Chinese."
    },
    {
        "label": "Spanglish (casual)",
        "text": "Oye, necesito hacer un update del software pero no sé cómo hacer el backup de mis files.",
        "task": "Understand this Spanglish text and provide a helpful response in Spanish."
    },
    {
        "label": "Hinglish (business)",
        "text": "Humein next quarter ke liye ek naya marketing strategy banana hai, especially social media engagement ke liye.",
        "task": "Understand this Hinglish (Hindi-English mix) text and respond helpfully in Hindi."
    },
]

for item in mixed_inputs:
    print(f"--- {item['label']} ---")
    print(f"Input: {item['text']}\n")

    messages = [
        {"role": "system", "content": item["task"]},
        {"role": "user", "content": item["text"]}
    ]
    response = generate(messages, max_new_tokens=300, temperature=0.5)
    print(f"Response:\n{response}\n")

In [ ]:
# --- Preserving Untranslatable Terms During Translation ---
# Some terms should NOT be translated: brand names, technical jargon,
# culturally-specific concepts, or terms with no equivalent.

texts_with_untranslatables = [
    {
        "text": "The new iOS update improves the Siri experience with on-device machine learning, better CoreML integration, and enhanced ARKit features.",
        "source": "English",
        "target": "Japanese",
        "note": "Preserve: iOS, Siri, CoreML, ARKit (Apple product names)"
    },
    {
        "text": "Cette startup utilise le framework React avec une architecture microservices déployée sur Kubernetes.",
        "source": "French",
        "target": "Chinese",
        "note": "Preserve: React, microservices, Kubernetes (tech terms)"
    },
    {
        "text": "After achieving satori during zazen meditation, she felt a deep sense of wabi-sabi in the old garden.",
        "source": "English",
        "target": "Spanish",
        "note": "Preserve: satori, zazen, wabi-sabi (Japanese cultural concepts)"
    },
]

for item in texts_with_untranslatables:
    print(f"--- {item['source']} → {item['target']} ---")
    print(f"Original: {item['text']}")
    print(f"Note: {item['note']}\n")

    messages = [
        {"role": "system", "content": (
            f"You are an expert translator from {item['source']} to {item['target']}. "
            "IMPORTANT: Do NOT translate proper nouns, brand names, technical framework names, "
            "or culturally-specific terms that have no direct equivalent. Keep them in their "
            "original form. After the translation, list which terms you preserved and why."
        )},
        {"role": "user", "content": item["text"]}
    ]
    response = generate(messages, max_new_tokens=400, temperature=0.3)
    print(f"Result:\n{response}\n")

### Why Models Handle Code-Switching Naturally

Multilingual models handle code-switching well for several reasons:

1. **Training data includes mixed text**: The internet is full of mixed-language content — social media posts, technical forums, bilingual documents. The model has seen millions of examples of code-switching during pretraining.

2. **Shared embedding space**: Since all languages exist in the same vector space, switching between languages mid-sentence doesn't cause a "context break" for the model. It smoothly processes tokens from different scripts.

3. **Attention is language-agnostic**: The self-attention mechanism tracks relationships between tokens regardless of their language. It can connect an English technical term to its Chinese context naturally.

> **Practical tip**: When working with code-switching input, explicitly tell the model what language to respond in. Otherwise, it may mirror the mixed style or default to the dominant language in the input.

## Improving Translation Quality and Cultural Sensitivity

Finally, let's create a prompt that focuses on maintaining cultural context and idioms in translation.

In [ ]:
cultural_texts = [
    {"source_lang": "English", "target_lang": "Japanese", "text": "It's raining cats and dogs."},
    {"source_lang": "French", "target_lang": "English", "text": "Je suis dans le pétrin."},
    {"source_lang": "Spanish", "target_lang": "German", "text": "Cuesta un ojo de la cara."}
]

for t in cultural_texts:
    messages = [
        {"role": "system", "content": (
            f"You are an expert translator from {t['source_lang']} to {t['target_lang']}. "
            "Pay special attention to cultural context and idiomatic expressions."
        )},
        {"role": "user", "content": (
            f"Translate the following text and provide:\n"
            f"1. A direct translation\n"
            f"2. A culturally adapted translation (if different)\n"
            f"3. Explanations of any cultural nuances or idioms\n\n"
            f"{t['source_lang']} text: {t['text']}"
        )}
    ]
    response = generate(messages)
    print(f"From {t['source_lang']} to {t['target_lang']}:")
    print(f"Original: {t['text']}")
    print(f"Translation and Explanation:\n{response}")
    print()

## Low-Resource Language Considerations

### The Long Tail of Languages

There are roughly 7,000 languages spoken worldwide, but LLMs are trained on a tiny fraction. Even among Qwen3's 29+ supported languages, performance varies dramatically. Let's measure this directly.

In [ ]:
# --- High-Resource vs Low-Resource Language Quality ---
# Test the same task across languages with different resource levels.

test_task = "Explain what photosynthesis is in exactly 3 sentences."

languages_to_test = {
    "English":    {"tier": "High",   "instruction": "Explain what photosynthesis is in exactly 3 sentences."},
    "Chinese":    {"tier": "High",   "instruction": "用恰好3句话解释什么是光合作用。"},
    "French":     {"tier": "Medium", "instruction": "Expliquez ce qu'est la photosynthèse en exactement 3 phrases."},
    "Korean":     {"tier": "Medium", "instruction": "광합성이 무엇인지 정확히 3문장으로 설명하세요."},
    "Vietnamese": {"tier": "Lower",  "instruction": "Giải thích quang hợp là gì trong đúng 3 câu."},
    "Thai":       {"tier": "Lower",  "instruction": "อธิบายว่าการสังเคราะห์แสงคืออะไรในประโยคเพียง 3 ประโยค"},
}

for lang, info in languages_to_test.items():
    print(f"{'='*60}")
    print(f"[{info['tier'].upper()} resource] {lang}")
    print(f"{'='*60}")

    messages = [
        {"role": "system", "content": f"You are a science teacher. Respond in {lang}."},
        {"role": "user", "content": info["instruction"]}
    ]
    response = generate(messages, max_new_tokens=300, temperature=0.3)
    print(f"\nPrompt: {info['instruction']}")
    print(f"\nResponse:\n{response}\n")

print("→ Observe: Higher-resource languages tend to produce more accurate,")
print("  fluent, and well-structured responses. Lower-resource languages may")
print("  show issues like: mixing in English words, less precise terminology,")
print("  or not following the '3 sentences' constraint as strictly.")

### Strategies for Lower-Resource Languages

When working with languages where the model has limited training data, these techniques can improve quality:

| Strategy | How It Works | When to Use |
|---|---|---|
| **English as pivot** | Translate input → English, process in English, translate output back | Complex reasoning tasks |
| **Few-shot examples** | Provide 2-3 examples of the desired input/output in the target language | When output format matters |
| **Simpler tasks** | Break complex tasks into simpler sub-tasks | When direct answers are unreliable |
| **English-first prompting** | Instruct in English, request output in target language | Almost always beneficial |
| **Post-editing pipeline** | Generate in high-resource language, then translate | When accuracy is critical |

> **Key insight**: The quality gap between high-resource and low-resource languages is **not fixed**. As models train on more diverse data with each generation, this gap narrows. Qwen3 supports many more languages than Qwen1 did. Always benchmark on your specific use case.

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** rewrite a prompt using the technique from this notebook. Document what changes and why.

**Exercise 2 — Build:** test the technique on a different task and compare results. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** combine this technique with one from a previous notebook.

## 📝 Key Takeaways

- **Overview** — revisit this section for reference
- **Motivation** — revisit this section for reference
- **Key Components** — revisit this section for reference
- **Method Details** — revisit this section for reference
- **Conclusion** — revisit this section for reference


## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the prompt-engineering/ directory

---

## 🧭 Navigation

⬅️ **Previous:** [Intro-Prompt-Engineering-Lesson](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/prompt-engineering/intro-prompt-engineering-lesson.ipynb) | ➡️ **Next:** [Negative-Prompting](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/prompt-engineering/negative-prompting.ipynb)